In [1]:
# NB_03_FHIR_Gold - Corrected Gold Layer
# ============================================================
# Reads ONLY current rows from Silver (is_current = True)
# Fixes JOIN keys by stripping "ResourceType/" prefix
# ============================================================

# === Fix for ancient dates (required in Fabric) ===
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")
print("✅ Datetime rebase config applied (LEGACY mode)\n")

print("🏆 Starting Gold Layer Creation...\n")

# ── 1. Dim Patient ───────────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE TABLE gold_dim_patient AS
    SELECT
        id                          AS patient_id,
        patient_name,
        gender,
        birthDate,
        address,
        valid_from,
        is_current,
        load_date,
        extraction_timestamp
    FROM silver_patient
    WHERE is_current = True          -- ✅ current records only
""")
print("✅ gold_dim_patient created")


# ── 2. Dim Encounter ─────────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE TABLE gold_dim_encounter AS
    SELECT
        id                                      AS encounter_id,
        status,
        class,
        period_start,
        period_end,
        split(subject.reference, '/')[1]        AS patient_id,   -- ✅ strip "Patient/" prefix
        type,
        valid_from,
        is_current,
        load_date
    FROM silver_encounter
    WHERE is_current = True                      -- ✅ current records only
""")
print("✅ gold_dim_encounter created")


# ── 3. Fact Observation ──────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE TABLE gold_fact_observation AS
    SELECT
        id                                      AS observation_id,
        split(subject.reference,  '/')[1]       AS patient_id,    -- ✅ strip "Patient/" prefix
        split(encounter.reference, '/')[1]      AS encounter_id,  -- ✅ strip "Encounter/" prefix
        code.text                               AS observation_code,
        valueQuantity.value                     AS value,
        valueQuantity.unit                      AS unit,
        valueQuantity.system                    AS unit_system,
        effectiveDateTime,
        status,
        valid_from,
        load_date
    FROM silver_observation
    WHERE is_current = True                      -- ✅ current records only
""")
print("✅ gold_fact_observation created")


# ── 4. Fact Condition ────────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE TABLE gold_fact_condition AS
    SELECT
        id                                      AS condition_id,
        split(subject.reference,  '/')[1]       AS patient_id,    -- ✅ strip "Patient/" prefix
        split(encounter.reference, '/')[1]      AS encounter_id,  -- ✅ strip "Encounter/" prefix
        code.text                               AS condition_code,
        clinicalStatus.coding[0].code           AS clinical_status,
        onsetDateTime,
        recordedDate,
        valid_from,
        load_date
    FROM silver_condition
    WHERE is_current = True                      -- ✅ current records only
""")
print("✅ gold_fact_condition created")


# ── Analytical Views ─────────────────────────────────────────────

# View 1: Patient + Encounter + Observation (wide clinical view)
spark.sql("""
    CREATE OR REPLACE VIEW gold_vw_patient_encounter_observation AS
    SELECT
        p.patient_id,
        p.patient_name,
        p.gender,
        p.birthDate,
        e.encounter_id,
        e.period_start                          AS encounter_start,
        e.period_end                            AS encounter_end,
        e.status                                AS encounter_status,
        o.observation_id,
        o.observation_code,
        o.value                                 AS observation_value,
        o.unit                                  AS observation_unit,
        o.effectiveDateTime                     AS observation_date
    FROM gold_dim_patient         p
    LEFT JOIN gold_dim_encounter  e  ON p.patient_id  = e.patient_id
    LEFT JOIN gold_fact_observation o ON e.encounter_id = o.encounter_id
""")
print("✅ gold_vw_patient_encounter_observation created")


# View 2: Patient + Conditions
spark.sql("""
    CREATE OR REPLACE VIEW gold_vw_patient_conditions AS
    SELECT
        p.patient_id,
        p.patient_name,
        p.gender,
        p.birthDate,
        c.condition_id,
        c.condition_code,
        c.clinical_status,
        c.onsetDateTime                         AS condition_onset,
        c.recordedDate                          AS condition_recorded
    FROM gold_dim_patient        p
    LEFT JOIN gold_fact_condition c  ON p.patient_id = c.patient_id
""")
print("✅ gold_vw_patient_conditions created")


# View 3: Full clinical summary (all 4 resources joined)
spark.sql("""
    CREATE OR REPLACE VIEW gold_vw_full_clinical_summary AS
    SELECT
        p.patient_id,
        p.patient_name,
        p.gender,
        p.birthDate,
        e.encounter_id,
        e.period_start                          AS encounter_date,
        e.status                                AS encounter_status,
        o.observation_code,
        o.value                                 AS observation_value,
        o.unit                                  AS observation_unit,
        c.condition_code,
        c.clinical_status,
        c.onsetDateTime                         AS condition_onset
    FROM gold_dim_patient               p
    LEFT JOIN gold_dim_encounter        e  ON p.patient_id   = e.patient_id
    LEFT JOIN gold_fact_observation     o  ON e.encounter_id = o.encounter_id
    LEFT JOIN gold_fact_condition       c  ON p.patient_id   = c.patient_id
""")
print("✅ gold_vw_full_clinical_summary created")


# ── Final Summary ─────────────────────────────────────────────────
print("\n" + "="*70)
print("🎉 GOLD LAYER CREATED SUCCESSFULLY!")
print("="*70)

tables = [
    "gold_dim_patient",
    "gold_dim_encounter",
    "gold_fact_observation",
    "gold_fact_condition"
]
views = [
    "gold_vw_patient_encounter_observation",
    "gold_vw_patient_conditions",
    "gold_vw_full_clinical_summary"
]

print("\n📊 Gold Tables:")
for t in tables:
    try:
        count = spark.table(t).count()
        print(f"   {t:50} → {count:,} rows")
    except Exception as e:
        print(f"   {t:50} → ❌ Error: {e}")

print("\n👁️  Gold Views:")
for v in views:
    try:
        count = spark.table(v).count()
        print(f"   {v:50} → {count:,} rows")
    except Exception as e:
        print(f"   {v:50} → ❌ Error: {e}")

print("\n✅ Connect Power BI to Gold tables/views above!")

StatementMeta(, 47b37f7f-b334-42bb-9600-674a5cff78f2, 3, Finished, Available, Finished, False)

✅ Datetime rebase configuration applied (LEGACY mode)

🏆 Starting Gold Layer Creation...

✅ gold_dim_patient created
✅ gold_dim_encounter created
✅ gold_fact_observation created
✅ gold_fact_condition created
✅ Analytical views created

🎉 GOLD LAYER CREATED SUCCESSFULLY!

Gold Tables:
   gold_dim_patient                              → 1,326 rows
   gold_dim_encounter                            → 1,500 rows
   gold_fact_observation                         → 1,500 rows
   gold_fact_condition                           → 1,500 rows

Gold Views:
   gold_vw_patient_encounter_observation         → 1,326 rows
   gold_vw_patient_conditions                    → 1,326 rows

✅ You can now connect Power BI to these Gold tables/views!
